# 01 · 黄金链端到端 (S1–S5 + M1)

Email → Context → RFQ → DFM → Quote → 辟牟援推止 → HITL/BLOCKED/REPLY → CRM+Memory。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
ctrl = build_controller()
scen = json.loads((ROOT/'data'/'golden_scenarios.json').read_text(encoding='utf-8'))['scenarios']
rows = []
for sc in scen:
    r = ctrl.run(email_text=sc['email'], customer=sc.get('customer'), voice_transcript=sc.get('voice_transcript'))
    ok = r['state']==sc['expect_state'] and r['verification_status']==sc['expect_status'] and r['audit_valid']
    rows.append((sc['id'], sc['expect_state'], r['state'], r['verification_status'],
                 (r['quote'] or {}).get('unit_price'), r['margin_pct'],
                 ','.join(r['multimodal_conflicts']) or '-', 'OK' if ok else 'XX'))
print('%-4s %-10s %-10s %-8s %10s %8s  %-22s %s' % ('ID','EXPECT','STATE','VERIFY','UNIT','MARGIN','CONFLICT',''))
for x in rows:
    print('%-4s %-10s %-10s %-8s %10s %8s  %-22s %s' % (x[0],x[1],x[2],x[3],x[4],x[5],x[6],x[7]))
print('ALL OK:', all(x[7]=='OK' for x in rows))

## 单条 context 全貌（S3 硬冲突 BLOCKED）

In [ ]:
s3 = next(s for s in scen if s['id']=='S3')
r3 = ctrl.run(email_text=s3['email'], customer=s3.get('customer'))
print('state=', r3['state'], '| verify=', r3['verification_status'])
print('DFM conflicts=', r3['dfm']['conflicts'])
print('reply subject=', r3['reply']['subject'])
print('auto_send=', r3['reply']['auto_send'], '| audit_valid=', r3['audit_valid'])